In [1]:
import pandas as pd
import re

# Load the raw reviews
df = pd.read_csv("trello_reviews_raw.csv")

# Re-clean exactly as in the exploration notebook:
# 1. drop reviews with no text, 2. drop duplicate review text
df = df.dropna(subset=["review_text"]).drop_duplicates(subset=["review_text"]).reset_index(drop=True)

print("Shape after cleaning:", df.shape)
df[["review_text", "rating"]].head()

Shape after cleaning: (4531, 7)


,review_text,rating
0,So far I love this app. Definitely not what it...,5
1,Last update ruined adding new items straight t...,1
2,Trello is a simple but powerful application. M...,5
3,too difficult and confusing to navigate the app,2
4,I use this app to manage my team for a project...,5


## TF-IDF preprocessing (heavy cleaning)

The traditional method matches reviews on exact shared words, so we clean deeply: lowercase, remove URLs/punctuation/numbers, remove filler stopwords, and stem words to their roots so variants like "crash / crashes / crashing" count as one.

In [3]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def clean_for_tfidf(text):
    text = text.lower()                                       # 1. lowercase everything
    text = re.sub(r"http\S+", " ", text)                      # 2. remove web links
    text = re.sub(r"[^a-z\s]", " ", text)                     # 3. keep only letters + spaces
    words = text.split()                                      # 4. break into words
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]  # 5. drop stopwords
    words = [w for w in words if len(w) > 1]                  # 6. drop stray single letters (s, t from contractions)
    words = [stemmer.stem(w) for w in words]                  # 7. stem to roots
    return " ".join(words)                                    # 8. rejoin

df["text_tfidf"] = df["review_text"].apply(clean_for_tfidf)
df[["review_text", "text_tfidf"]].head(10)

,review_text,text_tfidf
0,So far I love this app. Definitely not what it...,far love app definit design work great color c...
1,Last update ruined adding new items straight t...,updat ruin ad new item straight inbox quick wa...
2,Trello is a simple but powerful application. M...,trello simpl power applic make organ task mana...
3,too difficult and confusing to navigate the app,difficult confus navig app
4,I use this app to manage my team for a project...,use app manag team project amaz plan execut ta...
5,very useful,use
6,App somehow doesn't respect links to the websi...,app doesn respect link websit open browser mat...
7,#1Trello Can Turn You Into Millionaire or Mult...,trello turn millionair multi millionair help h...
8,I downloaded the app today to test it out for ...,download app today test meal planner decid del...
9,The service blocks email aliases for accounts....,servic block email alias account forc sacrific...


## Embedding preprocessing (light cleaning)

The semantic model was trained on natural human sentences and uses punctuation, casing, word order, and words like "not" to understand meaning. Aggressive cleaning would destroy that signal, so we only strip genuine junk (URLs, extra whitespace) and leave the rest intact.

In [7]:
def clean_for_embedding(text):
    text = re.sub(r"http\S+", " ", text)   # remove web links (still just noise)
    text = re.sub(r"\s+", " ", text)       # collapse extra spaces / line breaks into one space
    return text.strip()                    # trim leading/trailing spaces

df["text_embedding"] = df["review_text"].apply(clean_for_embedding)

# See all three side by side: original, heavy-cleaned, light-cleaned
df[["review_text", "text_tfidf", "text_embedding"]].head(10)

,review_text,text_tfidf,text_embedding
0,So far I love this app. Definitely not what it...,far love app definit design work great color c...,So far I love this app. Definitely not what it...
1,Last update ruined adding new items straight t...,updat ruin ad new item straight inbox quick wa...,Last update ruined adding new items straight t...
2,Trello is a simple but powerful application. M...,trello simpl power applic make organ task mana...,Trello is a simple but powerful application. M...
3,too difficult and confusing to navigate the app,difficult confus navig app,too difficult and confusing to navigate the app
4,I use this app to manage my team for a project...,use app manag team project amaz plan execut ta...,I use this app to manage my team for a project...
5,very useful,use,very useful
6,App somehow doesn't respect links to the websi...,app doesn respect link websit open browser mat...,App somehow doesn't respect links to the websi...
7,#1Trello Can Turn You Into Millionaire or Mult...,trello turn millionair multi millionair help h...,#1Trello Can Turn You Into Millionaire or Mult...
8,I downloaded the app today to test it out for ...,download app today test meal planner decid del...,I downloaded the app today to test it out for ...
9,The service blocks email aliases for accounts....,servic block email alias account forc sacrific...,The service blocks email aliases for accounts....


In [5]:
# How many reviews were completely emptied by the heavy TF-IDF cleaning?
empty_mask = df["text_tfidf"].str.strip() == ""
print("Reviews emptied by TF-IDF cleaning:", empty_mask.sum())

# Peek at a few of them — note their embedding version usually still has content
df[empty_mask][["review_text", "text_tfidf", "text_embedding"]].head(10)

Reviews emptied by TF-IDF cleaning: 71


,review_text,text_tfidf,text_embedding
102,👌,,👌
409,功能是很好用，但能不能給繁體中文的語言，我選繁體中文他還是跳簡體中文,,功能是很好用，但能不能給繁體中文的語言，我選繁體中文他還是跳簡體中文
413,👍,,👍
515,Топ,,Топ
546,Трелло як трелло,,Трелло як трелло
557,کگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگ...,,کگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگگ...
568,8 y,,8 y
603,Q &^ &. \555=3. . - 1. : .,,Q &^ &. \555=3. . - 1. : .
611,جميل وبسيط,,جميل وبسيط
616,Five out of five 5⃣,,Five out of five 5⃣


## Dropping emptied reviews

Heavy TF-IDF cleaning emptied 71 reviews (emoji-only, non-English, gibberish, or all-stopword text). These carry no usable signal for a word-based analysis. We drop them from the dataset so that **both** pipelines cluster the identical set of reviews, keeping the comparison fair. Final preprocessed set: 4,460 reviews.

In [6]:
# Drop reviews emptied by heavy cleaning, so BOTH methods cluster the same reviews
before = len(df)
df = df[df["text_tfidf"].str.strip() != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} emptied reviews. Remaining: {len(df)}")

# Save the preprocessed dataset for the clustering notebooks to load
df.to_csv("trello_reviews_preprocessed.csv", index=False)
print("Saved to trello_reviews_preprocessed.csv")

Dropped 71 emptied reviews. Remaining: 4460
Saved to trello_reviews_preprocessed.csv
